# SA tutor: Colab runner
Open this notebook in Google Colab. Run the cells from the top, **one at a time**, and read what each prints.
Everything lives on your Google Drive (Colab forgets everything when it disconnects).

**Before you start:** upload the whole `sa-tutor-kit` folder to Google Drive (My Drive). Put your `workspace/` inside it
(or copy `workspace/build/<round>` and `workspace/eval` there). Then in Colab: *Runtime -> Change runtime type -> T4 GPU*.

In [ ]:
# 1. Is there a GPU?
!nvidia-smi

## 2. Connect Google Drive and go to the kit folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
KIT = '/content/drive/MyDrive/sa-tutor-kit'          # <- change if you put it somewhere else
os.environ['SA_WORKSPACE'] = KIT + '/workspace'       # all scripts read/write here
%cd {KIT}
!ls

## 3. Install the tools (about 3-5 minutes)
If Unsloth's own Qwen3.5 notebook uses a different install line, **use theirs** (they update it often).

In [ ]:
!pip install -q unsloth
!pip install -q sympy pypdf matplotlib

## 4. Hugging Face login (only needed for models that ask you to accept a licence, e.g. Gemma)

In [ ]:
from huggingface_hub import login
login()

## 5. AUDITION base models (before any training)
Each run asks a candidate the same exam. Copy the **exact** model names from their Hugging Face pages.
Compare with `eval/report_eval.py` afterwards. Delete the lines for models you do not want to test.

In [ ]:
MODELS = {
    'base-qwen35-4b': 'Qwen/Qwen3.5-4B',
    # 'base-qwen38-4b-unofficial': '<paste the exact name from the Hugging Face page>',
    # 'base-gemma4-e4b': '<paste the exact name from the Hugging Face page>',
}
for name, model in MODELS.items():
    style = 'gemma' if 'gemma' in model.lower() else 'qwen'
    !python eval/run_eval.py --name {name} --backend hf --model {model} --load-4bit --think-style {style} --tags core --max-new-tokens 800
    !python eval/score_eval.py --name {name}

In [ ]:
!python eval/report_eval.py --runs base-qwen35-4b --out audition

## 6. TRAIN round 1
Step 6a is free of charge and needs no GPU time: it shows exactly what the model will be trained on. **Read the printed sample.**

In [ ]:
ROUND = 'r1'                                           # name of your build folder under workspace/build
MODEL = 'unsloth/Qwen3.5-4B'                           # the base model you chose
!python training/train_sft.py --model {MODEL} --data workspace/build/{ROUND} --dry-run --show 2

6b. **The "can it memorise?" test** (about 5-10 minutes). 20 samples, 15 passes. The loss at the end should be close to zero.
If it is not, something is wrong with the data or the template: do NOT start the long run.

In [ ]:
!python training/train_sft.py --model {MODEL} --data workspace/build/{ROUND} --out workspace/runs/overfit --limit 20 --epochs 15 --show 0

6c. **The real run.** Checkpoints are saved to Drive after every epoch. If Colab disconnects, run this cell again with `--resume` added.

In [ ]:
!python training/train_sft.py --model {MODEL} --data workspace/build/{ROUND} --out workspace/runs/{ROUND} --epochs 2 --show 0

## 7. EVALUATE the trained model (one command: answer, mark, compare, report)
`--compare` lists earlier runs (oldest first). `--ceiling` is the teacher's run on the same exam. Read `decision.md` and `failures.md`.

In [ ]:
!python eval/evaluate.py --name {ROUND} --backend hf --model {MODEL} --adapter workspace/runs/{ROUND}/adapter --compare base-qwen35-4b --ceiling teacher --analyze

## 8. More samples for consistency (5 answers per question at temperature 0.7)
A model that is right 5 times out of 5 is far more trustworthy than one that is right 3 times out of 5.

In [ ]:
!python eval/run_eval.py --name {ROUND}-n5 --backend hf --model {MODEL} --adapter workspace/runs/{ROUND}/adapter --tags core --n 5 --temperature 0.7
!python eval/score_eval.py --name {ROUND}-n5

## 9. EXPORT the files people run (several sizes from the same trained model)
Test every file with the exam before you ship it (see GUIDE.md phase 10).

In [ ]:
!python training/export_gguf.py --adapter workspace/runs/{ROUND}/adapter --out workspace/exports/tutor-4b-{ROUND} --quants q6_k q4_k_m

## 10. Preference training (DPO), later
After SFT passes the content checks but habits remain (leaks, language drift): build pairs with `training/make_dpo_pairs.py`
(GUIDE.md phase 9), then train with **Unsloth's official DPO notebook**, pointing its dataset cell at your pairs file
(`workspace/build/dpo_<round>.jsonl`, standard prompt / chosen / rejected format).